# Pipeline Results Viewer
Scenario A & B 처리 결과 확인 — 이미지 / 캡션 / 어노테이션 / DB 적재 현황

In [ ]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

_p = Path.cwd()
PIPELINE = _p if (_p / "flows").exists() else _p / "pipeline"
sys.path.insert(0, str(PIPELINE))
os.chdir(PIPELINE)
load_dotenv(PIPELINE.parent / ".env")
print("Working dir:", PIPELINE)

In [ ]:
from libsql import connect
from src.config import DB_URL, DB_ACCESS_TOKEN

conn = connect(DB_URL, auth_token=DB_ACCESS_TOKEN, _uri=True)

# 전체 아이템 수 확인
total = conn.execute("SELECT COUNT(*) FROM fashion_items").fetchone()[0]
by_status = conn.execute(
    "SELECT status, COUNT(*) FROM fashion_items GROUP BY status ORDER BY status"
).fetchall()

print(f"Total items in DB: {total}")
print("\nStatus breakdown:")
for status, cnt in by_status:
    print(f"  {status:<20} {cnt}건")

---
## 조회 대상 설정
확인할 file_id 목록을 지정하거나, 최근 처리된 N건을 조회합니다.

In [ ]:
# ── 조회 방법 선택 ──────────────────────────────────────────────────────────
# 방법 1: 특정 file_id 지정
TARGET_IDS = []   # 예: ["1028690", "1029079", "101858"]  (빈 리스트면 방법 2 사용)

# 방법 2: 최근 N건 조회
RECENT_N = 10

# 방법 3: status로 필터
FILTER_STATUS = None   # 예: "complete" / "vlm_only" / "pending_label" / None(전체)
# ────────────────────────────────────────────────────────────────────────────

if TARGET_IDS:
    placeholders = ",".join(["?"] * len(TARGET_IDS))
    rows = conn.execute(
        f"SELECT * FROM fashion_items WHERE file_id IN ({placeholders})",
        TARGET_IDS
    ).fetchall()
    col_names = [d[0] for d in conn.execute(
        f"SELECT * FROM fashion_items WHERE file_id IN ({placeholders}) LIMIT 0",
        TARGET_IDS
    ).description]
elif FILTER_STATUS:
    rows = conn.execute(
        "SELECT * FROM fashion_items WHERE status=? ORDER BY indexed_at DESC LIMIT ?",
        (FILTER_STATUS, RECENT_N)
    ).fetchall()
    col_names = [d[0] for d in conn.execute(
        "SELECT * FROM fashion_items LIMIT 0"
    ).description]
else:
    rows = conn.execute(
        "SELECT * FROM fashion_items ORDER BY indexed_at DESC LIMIT ?",
        (RECENT_N,)
    ).fetchall()
    col_names = [d[0] for d in conn.execute(
        "SELECT * FROM fashion_items LIMIT 0"
    ).description]

items = [dict(zip(col_names, r)) for r in rows]
print(f"{len(items)}건 로드됨")
for it in items:
    print(f"  {it['file_id']:<14} {it['category']:<12} {it['status']}")

---
## 결과 카드 출력
각 아이템의 마스킹 이미지, 수동 라벨, VLM 캡션, 분류기 결과를 한눈에 확인

In [ ]:
import json
import urllib.request
import io
from IPython.display import display, Image as IPImage, HTML
from PIL import Image

ARCHIVE_DIR = PIPELINE / "data" / "masked_images_archive"


def _load_image(item: dict):
    """로컬 아카이브 우선, 없으면 R2 URL에서 다운로드"""
    fid = item["file_id"]
    cat = item["category"]
    # 로컬 후보들
    candidates = list(ARCHIVE_DIR.rglob(f"{fid}_{cat}.jpg")) + \
                 list(ARCHIVE_DIR.rglob(f"{fid}_unknown.jpg"))
    if candidates:
        return Image.open(candidates[0])
    # R2 URL fallback
    url = item.get("image_url", "")
    if url:
        try:
            with urllib.request.urlopen(url, timeout=5) as resp:
                return Image.open(io.BytesIO(resp.read()))
        except Exception:
            pass
    return None


def _safe_json(val):
    if not val:
        return []
    try:
        return json.loads(val)
    except Exception:
        return val


def _status_badge(status: str) -> str:
    color = {"complete": "#2a9d8f", "vlm_only": "#e9c46a", "pending_label": "#e76f51"}.get(status, "#888")
    return f'<span style="background:{color};color:white;padding:2px 8px;border-radius:4px;font-size:12px;font-weight:bold">{status}</span>'


for item in items:
    fid     = item["file_id"]
    cat     = item["category"]
    status  = item["status"]

    # ── 헤더 ──────────────────────────────────────────────────────────────
    display(HTML(
        f'<hr><h3 style="margin-bottom:4px">{fid} &nbsp; '
        f'<code style="font-size:14px">{cat}</code> &nbsp; '
        f'{_status_badge(status)}</h3>'
    ))

    # ── 이미지 ────────────────────────────────────────────────────────────
    img = _load_image(item)
    if img:
        thumb = img.copy()
        thumb.thumbnail((200, 200))
        buf = io.BytesIO()
        thumb.save(buf, format="JPEG")
        display(IPImage(data=buf.getvalue()))
    else:
        display(HTML('<p style="color:#aaa">[이미지 없음]</p>'))

    # ── 수동 라벨 ─────────────────────────────────────────────────────────
    label_fields = [
        ("색상",   item.get("label_color")),
        ("서브색상", item.get("label_sub_color")),
        ("소재",   _safe_json(item.get("label_material"))),
        ("핏",    item.get("label_fit")),
        ("기장",   item.get("label_length")),
        ("소매",   item.get("label_sleeve")),
        ("넥라인",  item.get("label_neckline")),
        ("디테일",  _safe_json(item.get("label_detail"))),
        ("프린트",  _safe_json(item.get("label_print"))),
    ]
    filled = [(k, v) for k, v in label_fields if v and v != [] and v != "[]"]
    if filled:
        rows_html = "".join(
            f"<tr><td style='color:#555;padding:2px 8px'>{k}</td>"
            f"<td style='padding:2px 8px'><b>{v}</b></td></tr>"
            for k, v in filled
        )
        display(HTML(
            '<p style="margin:6px 0 2px;font-weight:bold;color:#2a9d8f">Manual Labels</p>'
            f'<table style="font-size:13px;border-collapse:collapse">{rows_html}</table>'
        ))
    else:
        display(HTML('<p style="color:#aaa;font-size:13px">Manual Labels: (없음 — VLM-only 또는 pending)</p>'))

    # ── 분류기 결과 ───────────────────────────────────────────────────────
    clf_fields = [
        ("패턴 크기",  item.get("pattern_size")),
        ("패턴 위치",  _safe_json(item.get("pattern_position"))),
        ("트림",     item.get("trim")),
        ("하의 기장",  item.get("bottom_length")),
        ("허리라인",   item.get("bottom_waist_rise")),
    ]
    clf_filled = [(k, v) for k, v in clf_fields if v and v != [] and v != "[]"]
    if clf_filled:
        rows_html = "".join(
            f"<tr><td style='color:#555;padding:2px 8px'>{k}</td>"
            f"<td style='padding:2px 8px'>{v}</td></tr>"
            for k, v in clf_filled
        )
        display(HTML(
            '<p style="margin:6px 0 2px;font-weight:bold;color:#457b9d">Classifier</p>'
            f'<table style="font-size:13px;border-collapse:collapse">{rows_html}</table>'
        ))

    # ── VLM 캡션 ──────────────────────────────────────────────────────────
    caption_cat   = item.get("caption_category", "")
    dense_caption = item.get("dense_caption", "")
    mood          = _safe_json(item.get("mood_and_tpo"))
    micro         = _safe_json(item.get("caption_micro_details"))
    flat_tags     = item.get("flat_tags", "")

    vlm_html = '<p style="margin:6px 0 2px;font-weight:bold;color:#6d4c41">VLM Caption</p>'
    if caption_cat:
        vlm_html += f'<p style="font-size:13px;margin:2px 0"><b>세부 카테고리:</b> {caption_cat}</p>'
    if dense_caption:
        vlm_html += f'<p style="font-size:13px;margin:2px 0"><b>dense_caption:</b> {dense_caption[:120]}...</p>'
    if mood:
        vlm_html += f'<p style="font-size:13px;margin:2px 0"><b>mood/TPO:</b> {", ".join(mood[:6])}</p>'
    if micro:
        vlm_html += f'<p style="font-size:13px;margin:2px 0"><b>micro details:</b> {", ".join(str(m) for m in micro[:5])}</p>'
    if flat_tags:
        display(HTML(vlm_html))
        tags_html = " ".join(
            f'<span style="background:#eee;border-radius:3px;padding:1px 5px;font-size:11px;margin:2px">{t}</span>'
            for t in flat_tags.split()[:20]
        )
        display(HTML(f'<p style="margin:2px 0;font-size:13px"><b>flat_tags:</b></p><div>{tags_html}</div>'))
    elif dense_caption or caption_cat:
        display(HTML(vlm_html))
    else:
        display(HTML('<p style="color:#aaa;font-size:13px">VLM: (캡션 없음)</p>'))

print("\n── 끝 ──")

---
## Summary Table

In [ ]:
import pandas as pd

summary = []
for it in items:
    summary.append({
        "file_id":       it["file_id"],
        "category":      it["category"],
        "status":        it["status"],
        "label_color":   it.get("label_color") or "-",
        "pattern_size":  it.get("pattern_size") or "-",
        "caption_cat":   (it.get("caption_category") or "-")[:30],
        "dense_caption": (it.get("dense_caption") or "-")[:50] + "...",
        "indexed_at":    (it.get("indexed_at") or "")[:16],
    })

df = pd.DataFrame(summary)

def _color_status(val):
    colors = {"complete": "background-color:#d4edda",
              "vlm_only": "background-color:#fff3cd",
              "pending_label": "background-color:#f8d7da"}
    return colors.get(val, "")

display(df.style.applymap(_color_status, subset=["status"]))

---
## Qdrant 벡터 색인 확인

In [ ]:
import os
from qdrant_client import QdrantClient

client = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])
info   = client.get_collection("visual")

print(f"Qdrant 'visual' 컬렉션")
print(f"  총 벡터 수:    {info.points_count}")
print(f"  indexed 수:   {info.indexed_vectors_count}")
print(f"  벡터 차원:    {info.config.params.vectors.size}")
print(f"  거리 함수:    {info.config.params.vectors.distance}")

In [ ]:
# 특정 file_id의 Qdrant 포인트 페이로드 확인
from flows.shared.tasks_qdrant import _make_point_id

for it in items[:3]:   # 처음 3건만
    fid = it["file_id"]
    cat = it["category"]
    pid = _make_point_id(fid, cat)
    results = client.retrieve(collection_name="visual", ids=[pid], with_payload=True)
    if results:
        p = results[0]
        print(f"\n[{fid}_{cat}]  point_id={p.id}")
        payload = p.payload or {}
        print(f"  category:     {payload.get('category')}")
        print(f"  label_color:  {payload.get('label_color')}")
        print(f"  dense_caption:{str(payload.get('dense_caption',''))[:60]}...")
        tags = (payload.get('flat_tags') or '').split()[:10]
        print(f"  flat_tags:    {' '.join(tags)}")
    else:
        print(f"[{fid}_{cat}]  Qdrant에 없음 (point_id={pid})")